# BDF1 CPU vs JAX/GPU for the Robertson Problem

This notebook compares a CPU baseline BDF1 solver against a vectorized JAX version designed for Colab GPU runs.

## 1) Imports and Dependencies

In [1]:
import numpy as np
from scipy.optimize import root
import time


## 2) JAX Setup Check (Colab GPU)

If JAX is missing on Colab, run this in a new cell first:
`%pip install -q -U "jax[cuda12]"`
Then restart the runtime.

In [2]:
JAX_AVAILABLE = True
try:
    import jax
    import jax.numpy as jnp
except ModuleNotFoundError:
    JAX_AVAILABLE = False
    jax = None
    jnp = None
    print("JAX is not installed. On Colab T4, run: %pip install -q -U \"jax[cuda12]\"")

if JAX_AVAILABLE:
    print("JAX version:", jax.__version__)
    print("JAX devices:", jax.devices())


JAX version: 0.7.2
JAX devices: [CudaDevice(id=0)]


## 3) Robertson System Definition (Stiff ODE)

In [3]:
def robertson(t, y):
    """Right-hand side of the Robertson three-component system."""
    y1, y2, y3 = y
    return np.array(
        [
            -0.04 * y1 + 1.0e4 * y2 * y3,
            0.04 * y1 - 1.0e4 * y2 * y3 - 3.0e7 * y2 * y2,
            3.0e7 * y2 * y2,
        ],
        dtype=y.dtype,
    )


## 4) CPU BDF1 Core Solver (Implicit Euler + Root Solve)

In [4]:
def _bdf1_step(f, t_n, y_n, h):
    """One BDF1 (implicit Euler) step solved with scipy.root."""

    def residual(y):
        return y - y_n - h * f(t_n + h, y)

    # Explicit Euler predictor gives root() a better starting point.
    y_guess = y_n + h * f(t_n, y_n)
    sol = root(residual, y_guess, method="hybr")
    if not sol.success:
        sol = root(residual, y_n, method="hybr")
    if not sol.success:
        raise RuntimeError(f"root() failed at t={t_n + h:.6g}: {sol.message}")
    return sol.x


def bdf1_integrate(f, y0, t0, t1, h):
    """Integrate y' = f(t, y) from t0 to t1 with fixed step h."""
    nt = int(np.ceil((t1 - t0) / h))
    t = t0 + h * np.arange(nt + 1)
    y = np.empty((nt + 1, y0.size), dtype=y0.dtype)
    y[0] = y0
    for n in range(nt):
        y[n + 1] = _bdf1_step(f, t[n], y[n], h)
    return t, y


## 5) CPU Batch Runner for Many Initial Conditions

In [5]:
def run_many_cpu(
    nconds,
    y0_base=None,
    t0=0.0,
    t1=1.0,
    h=1e-3,
    perturb_scale=0.0,
    seed=123,
):
    """Generate initial states and solve them sequentially on CPU."""
    if y0_base is None:
        y0_base = np.array([1.0, 0.0, 0.0])

    y0_base = np.asarray(y0_base, dtype=float)
    ics = np.repeat(y0_base[None, :], nconds, axis=0)

    if perturb_scale > 0.0:
        rng = np.random.default_rng(seed)
        ics = ics + perturb_scale * rng.standard_normal((nconds, 3))
        ics = np.clip(ics, 0.0, None)
        row_sum = ics.sum(axis=1, keepdims=True)
        row_sum[row_sum == 0.0] = 1.0
        ics = ics / row_sum

    results = []
    for y0 in ics:
        results.append(bdf1_integrate(robertson, y0, t0, t1, h))

    return ics, results


## 6) JAX/GPU Robertson Model

In [6]:
if JAX_AVAILABLE:
    @jax.jit
    def robertson_jax(t, y):
        y1, y2, y3 = y
        return jnp.stack(
            [
                -0.04 * y1 + 1.0e4 * y2 * y3,
                0.04 * y1 - 1.0e4 * y2 * y3 - 3.0e7 * y2 ** 2,
                3.0e7 * y2 ** 2,
            ]
        )


## 7) JAX BDF1 Integrator and Vectorized GPU Solver

In [7]:
if JAX_AVAILABLE:
    def _bdf1_step_jax(f, t_n, y_n, h, niter=8):
        # Fixed-point iteration (JIT-friendly) instead of scipy.root/Newton.
        y = y_n
        t_np1 = t_n + h
        for _ in range(niter):
            y = y_n + h * f(t_np1, y)
        return y


    def make_solve_many_gpu(t0=0.0, t1=1.0, h=1e-3, niter=8):
        # Keep these Python scalars/ints so scan length is static during JIT.
        nt = int(np.ceil((t1 - t0) / h))
        t0 = float(t0)
        h = float(h)
        niter = int(niter)

        def integrate_one(y0):
            def step(carry, _):
                t_n, y_n = carry
                y_np1 = _bdf1_step_jax(robertson_jax, t_n, y_n, h, niter=niter)
                return (t_n + h, y_np1), y_np1

            (_, _), ys = jax.lax.scan(step, (t0, y0), xs=None, length=nt)
            t = t0 + h * jnp.arange(nt + 1, dtype=y0.dtype)
            ys = jnp.vstack((y0[None, :], ys))
            return t, ys

        return jax.jit(jax.vmap(integrate_one))


## 8) Timing Utilities for CPU vs GPU Comparison

In [8]:
def compare_timings(nconds=32, t0=0.0, t1=0.1, h=1e-3, perturb_scale=0.0):
    """Run sequential CPU and parallel JAX/GPU versions and time both."""
    cpu_start = time.perf_counter()
    ics, cpu_res = run_many_cpu(
        nconds, t0=t0, t1=t1, h=h, perturb_scale=perturb_scale
    )
    cpu_time = time.perf_counter() - cpu_start

    if not JAX_AVAILABLE:
        raise RuntimeError("JAX is unavailable. Install JAX in Colab and rerun the notebook.")

    solve_many_gpu = make_solve_many_gpu(t0=t0, t1=t1, h=h)

    # Warm-up triggers one-time JIT compilation.
    _ = solve_many_gpu(jnp.zeros((1, 3), dtype=jnp.float32))

    conds = jnp.array(ics, dtype=jnp.float32)
    gpu_start = time.perf_counter()
    gpu_res = solve_many_gpu(conds)
    _ = jax.block_until_ready(gpu_res[1])
    gpu_time = time.perf_counter() - gpu_start

    return cpu_time, gpu_time, cpu_res, gpu_res


## 9) Example Execution

In [11]:
if JAX_AVAILABLE:
    cpu_t, gpu_t, _, _ = compare_timings(
        nconds=32, t0=0.0, t1=0.1, h=1e-3, perturb_scale=0.0
    )
    print(f"CPU (sequential)   : {cpu_t:.3f} s")
    print(f"GPU (jax vmap/jit) : {gpu_t:.3f} s")
else:
    print("Install JAX first, then rerun from Section 2 onward.")


CPU (sequential)   : 0.240 s
GPU (jax vmap/jit) : 0.326 s
